<a href="https://colab.research.google.com/github/BatoolAshour/PythonCodes/blob/main/DataAugmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [28]:
class simpleCNN(nn.Module):
  def __init__(self):
    super(simpleCNN, self).__init__()
    self.conv1= nn.Conv2d(3,32,kernel_size=3, padding=1)
    self.conv2= nn.Conv2d(32,64,kernel_size=3, padding=1)
    self.fc1= nn.Linear(64*8*8, 128)
    self.fc2= nn.Linear(128,10)
    self.pool= nn.MaxPool2d(2,2)
    self.relu= nn.ReLU()
    self.dropout= nn.Dropout(0.5)

  def forward(self, x):
    x = self.pool(self.relu(self.conv1(x)))
    x = self.pool(self.relu(self.conv2(x)))
    x= x.view(-1,64*8*8)
    x= self.relu(self.fc1(x))
    x= self.dropout(x)
    x= self.fc2(x)
    return x


In [29]:
def train_model(model, train_loader, criterion, optimizer, device):
  model.train()
  running_loss= 0.0
  for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs= model(images)
    loss= criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss+= loss.item()
  return running_loss / len(train_loader)

In [30]:
import multiprocessing

multiprocessing.cpu_count()

2

In [31]:
def load_data(data_agumentation=False):
  transforms_original = transforms.Compose([
      transforms.ToTensor(),
      transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
  ])
  transforms_augmented= transforms.Compose([
      transforms.RandomHorizontalFlip(),
      transforms.RandomRotation(10),
      transforms.RandomCrop(32,padding=4),
      transforms.ToTensor(),
      transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
  ])

  if data_agumentation:
    original_dataset= datasets.CIFAR10(root='./data',train=True,download=True, transform= transforms_original)
    augemented_dataset= datasets.CIFAR10(root='./data',train=True,download=True, transform= transforms_augmented)
    train_dataset= torch.utils.data.ConcatDataset([original_dataset, augemented_dataset])
  else:
    train_dataset= datasets.CIFAR10(root='./data',train=True,download=True, transform= transforms_original)

  test_dataset= datasets.CIFAR10(root='./data',train=False,download=True, transform= transforms_original)
  train_loader= DataLoader(train_dataset, batch_size=64, shuffle=True, num_workers=8)
  test_loader= DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=8)

  return train_loader, test_loader


In [32]:
device = torch.device('cuda' if torch.cuda.is_available()else 'cpu')
epochs= 2

train_loader, test_loader= load_data(data_agumentation=False)
model= simpleCNN().to(device)
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)

print('Without augmentation: ')
for epoch in range(epochs):
  loss= train_model(model, train_loader, criterion, optimizer, device)
  print(f'Epoch: {epoch}: Loss= {loss}')


accuracy= evaluate_model(model, test_loader, device)
print(f'Accuracy: {accuracy}%')


train_loader, test_loader= load_data(data_agumentation=True)
model= simpleCNN().to(device)
criterion= nn.CrossEntropyLoss()
optimizer= optim.Adam(model.parameters(), lr=0.001)

print('With augmentation: ')
for epoch in range(epochs):
  loss= train_model(model, train_loader, criterion, optimizer, device)
  print(f'Epoch: {epoch}: Loss= {loss}')


accuracy= evaluate_model(model, test_loader, device)
print(f'Accuracy: {accuracy}%')

Without augmentation: 


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Epoch: 0: Loss= 1.5601142041213678
Epoch: 1: Loss= 1.256211537076994
Accuracy: 63.5%
With augmentation: 
Epoch: 0: Loss= 1.4903944675463252
Epoch: 1: Loss= 1.188371599635785
Accuracy: 67.93%
